In [26]:
import numpy as np
import pandas as pd
import re

df=pd.read_csv('employee_records[1].csv')

# Columns
df=df.rename(columns={
    'age': 'Age',
    'GENDER': 'Gender',
    'email_address': 'EmailAddress',
    'salary': 'Salary',
    'dept': 'Department'
})

# Name
df['FullName']=df['FullName'].fillna("None")
df['FullName']=df['FullName'].str.lower()
df['FullName']=df['FullName'].str.title()

# Age
df['Age']=df['Age'].fillna(df['Age'].median())
df['Age']=df['Age'].round(0).astype(int)

# Gender
df['Gender']=df['Gender'].str.lower()
df['Gender']=df['Gender'].map({'male': 'M', 'female': 'F'})
df['Gender']=df['Gender'].fillna("Unknown")

# Email
email_pattern = r"^(?!.*\.\.)[\w\.-]+@[\w-]+(\.[\w-]+)+$"
df['EmailAddress']=df['EmailAddress'].apply(lambda x: x if re.match(email_pattern, str(x)) else np.nan)
df['EmailMissing'] = df['EmailAddress'].isnull()

# Salary
df['Salary']=pd.to_numeric(df['Salary'], errors='coerce')
df['Salary']=df['Salary'].fillna(df['Salary'].median())

# StartDate
def try_parse_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    
    # Remove spaces
    date_str = str(date_str).strip()

    # ISO format: YYYY-MM-DD or YYYY/MM/DD
    iso_pattern = r"^\d{4}[-/]\d{2}[-/]\d{2}$"

    try:
        if re.match(iso_pattern, date_str):
            # Parse normally (ISO)
            return pd.to_datetime(date_str, errors='coerce')
        else:
            # Try day-first formats (like 15-02-2020)
            return pd.to_datetime(date_str, errors='coerce', dayfirst=True)
    except:
        return pd.NaT

df['StartDate']=df['StartDate'].apply(try_parse_date)
df['StartDate'] = df['StartDate'].fillna(df['StartDate'].median())

# Department
df['Department']=df['Department'].str.strip().str.lower()
df=pd.get_dummies(df, columns=['Department'], drop_first=True)
df

# Others
df.loc[2, 'FullName']='Mark Zuckerberg'
df=df.drop_duplicates()
df=df.drop(index=[3])
df

,FullName,Age,Gender,EmailAddress,Salary,StartDate,EmailMissing,Department_hr,Department_marketing,Department_operations,Department_product
0,John Smith,27,M,NaN,55000.00,2020-01-15,True,False,False,False,False
1,Jane Doe,34,F,jane.doe@example.com,58000.00,2020-02-15,False,True,False,False,False
2,Mark Zuckerberg,35,M,NaN,95000.00,2020-03-31,True,False,False,False,True
4,Elon Musk,48,M,elon@tesla.com,58000.00,2022-12-03,False,False,False,False,False
6,Sarah O'Connor,33,F,NaN,68000.00,2020-05-15,True,False,False,True,False
7,Lana Lang,29,F,NaN,57000.00,2021-06-01,True,True,False,False,False
8,Peter Parker,34,M,spiderman@web.org,58000.00,2020-03-31,False,False,True,False,False
9,Steve Jobs,54,M,steve@apple.com,150000.55,2018-07-30,False,False,False,False,True
